# DMS — Train a YOLOv8 detector (cigarette / seatbelt / etc.) on a free GPU

This notebook trains an in-cabin object detector for the Driver Monitoring System.
The stock COCO model already handles **phone / food / drink**; here you train the
classes COCO lacks: **cigarette, seatbelt/no_seatbelt, sunglasses**.

**Before you start:** set the runtime to GPU — `Runtime → Change runtime type → T4 GPU`.
Then run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 3. Get a labelled dataset

Easiest path: pick a dataset on **Roboflow Universe** (search e.g. "cigarette",
"seatbelt", "driver distraction"). On the dataset page click **Download → YOLOv8 →
show download code**, then paste that snippet below (it fills in your workspace,
project, version, and API key). A free Roboflow account gives you an API key at
`https://app.roboflow.com/settings/api`.

Make sure the dataset's class names are ones the DMS understands (see
`SAFETY_CLASS_MAP` in `src/objects.py`): e.g. `cigarette`, `seatbelt`,
`no_seatbelt`, `sunglasses`, `phone`, `food`, `drink`.

In [ ]:
# --- PASTE YOUR ROBOFLOW SNIPPET BELOW (edit the placeholders) ---
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print("dataset:", DATA_YAML)

**Alternative — your own dataset.** If you already have a YOLO-format dataset
(images/ + labels/ + a data.yaml), zip it, upload via the Files panel, unzip,
and set `DATA_YAML` to its `data.yaml` path instead of running the cell above:

```python
# !unzip -q /content/my_dataset.zip -d /content/data
# DATA_YAML = "/content/data/data.yaml"
```

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano = fast; use yolov8s.pt for a bit more accuracy
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,   # early-stop if val stops improving
)

## 5. Check accuracy

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

## 6. Quick visual test (optional) — upload an image to try it on

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()  # pick a test image
for name in uploaded:
    pred = model.predict(name, conf=0.35, save=True)
    print("boxes:", len(pred[0].boxes))
    display(Image.open(pred[0].save_dir + "/" + name))

## 7. Download the trained weights

In [ ]:
from google.colab import files

best = "runs/detect/train/weights/best.pt"
print("Downloading", best)
files.download(best)

## 8. Plug it into the DMS

1. Put the downloaded `best.pt` in your project (e.g. `models/best.pt`).
2. In `src/config.py`, set:
   ```python
   object_model_path: str = "models/best.pt"
   ```
3. Run `python main.py`. Detections whose class names are in `SAFETY_CLASS_MAP`
   (cigarette, seatbelt, sunglasses, phone, food, drink) are drawn and flagged
   automatically.

**Tips:** more data + variety beats more epochs. Aim for a few thousand labelled
instances per class, and always validate on footage from **your** camera angle.